# Silver Layer, Customer Data Processing
**Source:** Bronze Delta Table `bronze.bronze_customer`  
**Target:** Silver Delta Table `/delta/silver/customers` (Hive table `silver.silver_customers`)  
**Pattern:** Incremental Load with Merge (UPSERT) based on `customer_id`  
**Transformations:**
- Validate age (18 – 100), non‑null email, non‑negative total_purchases
- Derive `customer_segment` from `total_purchases`
- Calculate `days_since_registration`
- Add `last_updated` timestamp
  
**Layer:** Silver

In [3]:
import os, sys, logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, expr

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "bronze_db"      : "bronze",
    "bronze_table"   : "bronze_customer",
    "silver_db"      : "silver",
    "silver_table"   : "silver_customers",
    "silver_path"    : "hdfs://master:8020/delta/silver/customers",   
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Silver_Customers_Load")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def create_silver_table_if_not_exists(spark):
    """Create the silver customers table if it doesn't exist."""
    spark.sql(f"""
        CREATE DATABASE IF NOT EXISTS {CONFIG['silver_db']}
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONFIG['silver_db']}.{CONFIG['silver_table']} (
            customer_id STRING,
            name STRING,
            email STRING,
            country STRING,
            customer_type STRING,
            registration_date DATE,
            age INT,
            gender STRING,
            total_purchases INT,
            customer_segment STRING,
            days_since_registration INT,
            last_updated TIMESTAMP
        )
        USING DELTA
        LOCATION '{CONFIG['silver_path']}'
    """)
    logger.info(f"Ensured table {CONFIG['silver_db']}.{CONFIG['silver_table']} exists.")

def get_last_processed_timestamp(spark):
    """Return the maximum last_updated timestamp from silver table, or a default."""
    try:
        df = spark.sql(f"SELECT MAX(last_updated) AS last_processed FROM {CONFIG['silver_db']}.{CONFIG['silver_table']}")
        row = df.collect()[0]
        ts = row['last_processed']
        if ts is None:
            return "1900-01-01T00:00:00.000+00:00"
        else:
            
            return ts.isoformat() if hasattr(ts, 'isoformat') else str(ts)
    except Exception as e:
        logger.warning(f"Could not read last_processed (table may be empty): {e}")
        return "1900-01-01T00:00:00.000+00:00"

def run():
    logger.info("Silver Customers pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    # Ensure silver table exists
    create_silver_table_if_not_exists(spark)

    # Get last processed timestamp
    last_processed = get_last_processed_timestamp(spark)
    logger.info(f"Last processed timestamp: {last_processed}")

    # Create temporary view of incremental bronze data
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW bronze_incremental AS
        SELECT *
        FROM {CONFIG['bronze_db']}.{CONFIG['bronze_table']}
        WHERE ingestion_timestamp > TIMESTAMP '{last_processed}'
    """)

    
    count_new = spark.table("bronze_incremental").count()
    if count_new == 0:
        logger.info("No new data to process. Exiting.")
        return
    logger.info(f"Found {count_new} new rows in bronze.")

    
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW silver_incremental AS
        SELECT
            customer_id,
            name,
            email,
            country,
            customer_type,
            registration_date,
            age,
            gender,
            total_purchases,
            CASE
                WHEN total_purchases > 10000 THEN 'High Value'
                WHEN total_purchases > 5000 THEN 'Medium Value'
                ELSE 'Low Value'
            END AS customer_segment,
            DATEDIFF(CURRENT_DATE(), registration_date) AS days_since_registration,
            CURRENT_TIMESTAMP() AS last_updated
        FROM bronze_incremental
        WHERE
            age BETWEEN 18 AND 100
            AND email IS NOT NULL
            AND total_purchases >= 0
    """)

    
    logger.info("Sample of transformed silver data:")
    spark.table("silver_incremental").show(5, truncate=False)

    # Merge into silver table
    merge_result = spark.sql(f"""
        MERGE INTO {CONFIG['silver_db']}.{CONFIG['silver_table']} AS target
        USING silver_incremental AS source
        ON target.customer_id = source.customer_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    # Log merge stats
    logger.info(f"Merge completed: {merge_result.first()['num_affected_rows']} rows affected.")

    
    final_count = spark.table(f"{CONFIG['silver_db']}.{CONFIG['silver_table']}").count()
    logger.info(f"SUCCESS: Silver customers table now has {final_count} rows.")

if __name__ == "__main__":
    run()

2026-03-19 11:49:49,657 [INFO] Silver Customers pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d2d39615-2887-4c95-a733-5ac2e6cfa97e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (4611ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (153ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (413ms)
:: resolution report :: resolve 3823ms